<a href="https://colab.research.google.com/github/raven-in-space/data-science-cohort-20/blob/main/Project-3/Project_3_SQL_Analysis_of_Singer_Songwriters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Analysis of Singer-Songwriter Sales by Media Type


* **Date:** `5/7/2026`
* **Author:** `Raven Otero-Symphony`

This project was completed as part of the CNM Ingenuity Data Science Bootcamp.

This project uses SQL to investigate the `chinook` music dataset, using a minimum of 13 SQL commands to demonstrate proficiency.

Methods used:

1. `SELECT`
2. `GROUP BY`
3. `ORDER BY`
4. `WHERE`
5. `DISTINCT`
6. `COUNT`
7. `BETWEEN`
8. `AND`
9. `LIMIT`
10. `MAX`
11. `MIN`
12. `SUM`
13. `AVG`

Let's start by importing standard libraries, installing `sqlite`, downloading and unzipping the `chinook` database.

In [2]:
import pandas as pd
import sqlite3 as db
from google.colab import output

In [3]:
# install sqlite package for Ubuntu
%%capture
%%bash
apt-get update
apt-get install -y sqlite3
pip install sqlite-web

In [4]:
# download chinook dataset
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip

Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [5]:
# unzip the zipped file
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [6]:
# list all folders in the directory
!ls -la

total 1180
drwxr-xr-x 1 root root   4096 May  8 15:38 .
drwxr-xr-x 1 root root   4096 May  8 14:41 ..
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 May  8 15:38 chinook.zip
drwxr-xr-x 4 root root   4096 Apr 16 13:28 .config
drwxr-xr-x 1 root root   4096 Apr 16 13:28 sample_data


In [7]:
!ls -lh

total 1.2M
-rw-r--r-- 1 root root 864K Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 299K May  8 15:38 chinook.zip
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data


In [8]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables
;

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [9]:
# connect the database to a variable for later calling
## recall "db" stands for the sqlite library
db_con = db.connect("chinook.db")

**Question:** How many tables are there? Confirm the data reflects what is shown in the [ERD](https://www.sqlitetutorial.net/sqlite-sample-database/).

In [ ]:
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


Next, let's take a look at the specific tables.

**Question:** What does the `tracks` database look like?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
LIMIT 10
;

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

**Question:** What does the `artists` database look like, and how does it differ from the `tracks` database?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM artists
limit 10
;

ArtistId  Name                
--------  --------------------
1         AC/DC               
2         Accept              
3         Aerosmith           
4         Alanis Morissette   
5         Alice In Chains     
6         Antônio Carlos Jobim
7         Apocalyptica        
8         Audioslave          
9         BackBeat            
10        Billy Cobham        


**Question:** What does the `albums` database look like?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM albums
LIMIT 10
;

AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       
6        Jagged Little Pill                     4       
7        Facelift                               5       
8        Warner 25 Anos                         6       
9        Plays Metallica By Four Cellos         7       
10       Audioslave                             8       


We see that there is a connection between `albums`, `artists`, and `tracks` through the "ArtistId" and "AlbumId" keys.

From `pandas` work earlier (removed to meet the SQL requirement for this project), I noticed that my favorite composer, "Edward Elgar" is listed as a `composer` in the `tracks` database.

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
WHERE Composer = 'Edward Elgar'
LIMIT 10
;

TrackId  Name                                                                       AlbumId  MediaTypeId  GenreId  Composer      Milliseconds  Bytes    UnitPrice
-------  -------------------------------------------------------------------------  -------  -----------  -------  ------------  ------------  -------  ---------
3421     Nimrod (Adagio) from Variations On an Original Theme, Op. 36 "Enigma"      290      2            24       Edward Elgar  250031        4124707  0.99     
3440     Concerto for Cello and Orchestra in E minor, Op. 85: I. Adagio - Moderato  306      2            24       Edward Elgar  483133        7865479  0.99     


**Question:** Is Edward Elgar listed as a `Name` in the `artists` database?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM artists
WHERE Name = 'Edward Elgar'
LIMIT 10
;

We see that Edward Elgar isn't found in the `artists` database. I now hypothesize that, since Elgar is a composer from the 19th-century, only more contemporary artists are listed.

**Question:** Is Michael Jackson listed as a composer in any of his songs?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
WHERE Composer = 'Michael Jackson'
LIMIT 10
;

TrackId  Name         AlbumId  MediaTypeId  GenreId  Composer         Milliseconds  Bytes    UnitPrice
-------  -----------  -------  -----------  -------  ---------------  ------------  -------  ---------
3382     Billie Jean  270      2            23       Michael Jackson  281401        4606408  0.99     


Let's save the investigation between "Composer" and "Name" for now, and look at another relationship: the `media_types` and `tracks`.

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM media_types
LIMIT 10
;

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   
5            AAC audio file             


In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
LIMIT 10
;

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

**Question:** What are the most common `MediaTypeId`'s by `track`?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT
  MediaTypeId,
  COUNT(*) AS track_count
FROM tracks
GROUP BY MediaTypeId
ORDER BY track_count DESC
;

MediaTypeId  track_count
-----------  -----------
1            3034       
2            237        
3            214        
5            11         
4            7          


Interesting...

`MediaTypeId` "1" (or "MPEG audio file") is the most popular choice by far, while "4" (or "Purchased AAC audio file") is the *least* popular.

**Question:** Which songs were purchased as an AAC audio file?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
WHERE MediaTypeId = 4
;

TrackId  Name                                                                            AlbumId  MediaTypeId  GenreId  Composer                  Milliseconds  Bytes     UnitPrice
-------  ------------------------------------------------------------------------------  -------  -----------  -------  ------------------------  ------------  --------  ---------
3336     War Pigs                                                                        260      4            23                                 234013        8052374   0.99     
3414     Symphony No. 104 in D Major "London": IV. Finale: Spiritoso                     283      4            24       Franz Joseph Haydn        306687        10085867  0.99     
3452     SCRIABIN: Prelude in B Major, Op. 11, No. 11                                    318      4            24                                 101293        3819535   0.99     
3479     Prometheus Overture, Op. 43                                                     324      4 

Somewhat unsurprisingly, symphonic pieces are found here. But what in the world is "War Pigs"?

Let's take a look at the `media_types` inbetween.

**Question:** Which tracks were purchased as protected files?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
WHERE MediaTypeId BETWEEN 2 AND 3
;

TrackId  Name                                                                                                           AlbumId  MediaTypeId  GenreId  Composer                                                                                                                                                                                      Milliseconds  Bytes       UnitPrice
-------  -------------------------------------------------------------------------------------------------------------  -------  -----------  -------  --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  ------------  ----------  ---------
2        Balls to the Wall                                                                                              2        2            1                                                                                                       

Let's revisit performers who are also songwriters (that is, musicians listed as both "artists" and "composers").

**Question:** How many unique `Name`'s are there in the `artists` database?

*Functions used:* `DISTINCT`, `ORDER BY`, `SELECT`

In [ ]:
%%script sqlite3 --column --header chinook.db
select DISTINCT name
from artists
ORDER BY name ASC
;

Name                                                                                 
-------------------------------------------------------------------------------------
A Cor Do Som                                                                         
AC/DC                                                                                
Aaron Copland & London Symphony Orchestra                                            
Aaron Goldberg                                                                       
Academy of St. Martin in the Fields & Sir Neville Marriner                           
Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner          
Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair
Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart            
Academy of St. Martin in the Fields, Sir Neville Marriner & William Bennett          
Accept                                                

Here, we find there are 275 unique artists listed.

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT COUNT(name)
FROM artists
;

COUNT(name)
-----------
275        


*However*... this is not completely accurate. We can see from the listed query that artists contain special characters such as `-;&ö` and others.

Let's take a look at the "Composers" to prove this point.

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT composer
FROM tracks
ORDER BY composer ASC
;

Composer                                                                                                                                                                                    
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                                                                                                                                            
A. F. Iommi, W. Ward, T. Butler, J. Osbourne                                                                                                                                                
A. Jamal                                                                                                                                                                                    
A.Bouchard/J.Bouchard/S.Pearlman                       

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT COUNT(composer)
FROM tracks
;

COUNT(composer)
---------------
2525           


**Question:** How many artists composed their own music?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM artists, tracks
WHERE artists.name = tracks.composer
ORDER BY name ASC
;

ArtistId  Name                   TrackId  Name                                                       AlbumId  MediaTypeId  GenreId  Composer               Milliseconds  Bytes     UnitPrice
--------  ---------------------  -------  ---------------------------------------------------------  -------  -----------  -------  ---------------------  ------------  --------  ---------
1         AC/DC                  18       Bad Boy Boogie                                             4        1            1        AC/DC                  267728        8776140   0.99     
1         AC/DC                  16       Dog Eat Dog                                                4        1            1        AC/DC                  215196        7032162   0.99     
1         AC/DC                  15       Go Down                                                    4        1            1        AC/DC                  331180        10847611  0.99     
1         AC/DC                  21       Hell Ain't A 

In [ ]:
%%script sqlite3 --column --header chinook.db
WITH legends AS(
  SELECT *
  FROM artists, tracks
  WHERE artists.name = tracks.composer
)
SELECT COUNT(*) AS total_rows
FROM legends
;

total_rows
----------
402       


### Aggregates


Let's move on to see if there are any financial trends for performing singer-songwriters, versus those who *didn't*.

We find that there is a relationship between `invoice_items` and `tracks`

#### MAX


In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM invoice_items
LIMIT 5
;

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity
-------------  ---------  -------  ---------  --------
1              1          2        0.99       1       
2              1          4        0.99       1       
3              2          6        0.99       1       
4              2          8        0.99       1       
5              2          10       0.99       1       


**Question:** What is the maximum unit price for a song?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT MAX(UnitPrice)
FROM invoice_items
;

MAX(UnitPrice)
--------------
1.99          


**Question:** What is the minimum?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT MIN(UnitPrice)
FROM invoice_items
;

MIN(UnitPrice)
--------------
0.99          


**Question:** How many sales are there total in the `invoice_items`?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT SUM(UnitPrice)
FROM invoice_items
;

SUM(UnitPrice)  
----------------
2328.59999999996


**Question:** Does this number match the sales in `tracks`?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT SUM(UnitPrice)
FROM tracks
;

SUM(UnitPrice) 
---------------
3680.9699999997


We find a discrepancy in the sum of `UnitPrice` between both `invoice_items` and `tracks`, likely because one serves as a "source of truth" and the other is a reference.

Let's take a look at the `invoices` table for clues.

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM invoices
;

InvoiceId  CustomerId  InvoiceDate          BillingAddress                            BillingCity          BillingState  BillingCountry  BillingPostalCode  Total
---------  ----------  -------------------  ----------------------------------------  -------------------  ------------  --------------  -----------------  -----
1          2           2009-01-01 00:00:00  Theodor-Heuss-Straße 34                   Stuttgart                          Germany         70174              1.98 
2          4           2009-01-02 00:00:00  Ullevålsveien 14                          Oslo                               Norway          0171               3.96 
3          8           2009-01-03 00:00:00  Grétrystraat 63                           Brussels                           Belgium         1000               5.94 
4          14          2009-01-06 00:00:00  8210 111 ST NW                            Edmonton             AB            Canada          T6G 2C7            8.91 
5          23          2009-

**Question:** What is the total sales amount by media type?

In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT
    tracks.MediaTypeId,
    SUM(invoice_items.UnitPrice * invoice_items.Quantity) AS total_sales
FROM invoice_items
JOIN tracks ON invoice_items.TrackId = tracks.TrackId
GROUP BY tracks.MediaTypeId
ORDER BY total_sales DESC
;

MediaTypeId  total_sales     
-----------  ----------------
1            1956.24000000002
3            220.89          
2            144.54          
4            3.96            
5            2.97            


As expected, we find that the most sales went toward the popular "MPEG audio file." Let's plot this.

In [ ]:
# store query as a variable
query = """
SELECT
    tracks.MediaTypeId,
    SUM(invoice_items.UnitPrice * invoice_items.Quantity) AS total_sales
FROM invoice_items
JOIN tracks ON invoice_items.TrackId = tracks.TrackId
GROUP BY tracks.MediaTypeId
ORDER BY total_sales DESC
;
"""

# assign to call later
media_type_sales = pd.read_sql_query( query, db_con )
media_type_sales

,MediaTypeId,total_sales
0,1,1956.24
1,3,220.89
2,2,144.54
3,4,3.96
4,5,2.97


In [ ]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM media_types
LIMIT 10
;

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   
5            AAC audio file             


In [ ]:
# add new col with readable labels
media_type_sales["media_names"] = [
    "MPEG audio file",
    "Protected AAC audio file",
    "Protected MPEG-4 video file",
    "Purchased AAC audio file",
    "AAC audio file"]

media_type_sales["media_shorthand"] = [
    "MPEG",
    "*AAC",
    "*MPEG-4",
    "Purchased AAC",
    "AAC"]

media_type_sales

,MediaTypeId,total_sales,media_names,media_shorthand
0,1,1956.24,MPEG audio file,MPEG
1,3,220.89,Protected AAC audio file,*AAC
2,2,144.54,Protected MPEG-4 video file,*MPEG-4
3,4,3.96,Purchased AAC audio file,Purchased AAC
4,5,2.97,AAC audio file,AAC


In [ ]:
import plotly.express as px

fig = px.bar(media_type_sales,
             x = "media_shorthand",
             y = "total_sales",
             text_auto = ".2s",
             title = "Chinook Total Sales by Media Type",
             labels={'media_shorthand': 'Media Type (*denotes protected file)',
                     'total_sales': 'Total Sales ($)'})
fig.show()

### 👩🏼‍💻 Future Development
* Clean the "Artist" and "Composer" fields for more accurate matching.
* Investigate which performing singer-songwriters made the most money.